# Gym Group — data exploration

Fetches check-in history from The Gym Group API and explores the data.

Requires `GYM_GROUP_USERNAME` and `GYM_GROUP_PASSWORD` in `.env`.

In [ ]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

import os
from pipeline.extractors.gymgroup import _fetch, _to_df

username = os.environ["GYM_GROUP_USERNAME"]
password = os.environ["GYM_GROUP_PASSWORD"]

In [ ]:
check_ins = _fetch(username, password)
print(f"{len(check_ins)} check-ins fetched")
check_ins[:2]

In [ ]:
df = _to_df(check_ins)
print(df.shape)
df.head(10)

In [ ]:
# Date range
print("From:", df["date"].min())
print("To:  ", df["date"].max())
print("Visits:", len(df))
print("Avg duration (min):", df["value"].mean().round(1))

In [ ]:
# Visits per gym location
df.group_by("category").len().sort("len", descending=True)

## SQL queries on R2 tables

Uses DuckDB to query parquet files directly from R2 via the S3-compatible API.

In [15]:
import duckdb
from dotenv import dotenv_values

env = dotenv_values("../.env.local.example")

conn = duckdb.connect()
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")

endpoint = (
    env["R2_ENDPOINT_URL"]
    .removeprefix("https://")
    .removeprefix("http://")
    .rstrip("/")
)

use_ssl = env["R2_ENDPOINT_URL"].startswith("https://")

conn.execute(f"""
    SET s3_endpoint = '{endpoint}';
    SET s3_access_key_id = '{env["R2_ACCESS_KEY_ID"]}';
    SET s3_secret_access_key = '{env["R2_SECRET_ACCESS_KEY"]}';
    SET s3_region = 'auto';
    SET s3_url_style = 'path';
    SET s3_use_ssl = {str(use_ssl).lower()};
""")

df = conn.execute("""
    SELECT *
    FROM read_parquet(
        's3://year-in-data/tables/gymgroup_visits.parquet'
    )
    ORDER BY date DESC
    LIMIT 20
""").df()

print(df)

         date category  duration_ms
0  2026-04-10   Bolton   17280000.0
1  2026-04-08   Bolton   16320000.0
2  2026-04-01   Bolton   13920000.0
3  2026-03-27   Bolton   20880000.0
4  2026-03-25   Bolton   22320000.0
5  2026-03-23   Bolton   13200000.0
6  2026-03-19   Bolton   16560000.0
7  2026-03-17   Bolton   17040000.0
8  2026-03-08   Bolton   15840000.0
9  2026-03-06   Bolton   18960000.0
10 2026-03-05   Bolton   20640000.0
11 2026-02-27   Bolton   12720000.0
12 2026-02-23   Bolton   17520000.0
13 2026-02-20   Bolton   16560000.0
14 2026-02-19   Bolton   17040000.0
15 2026-02-15   Bolton   18480000.0
16 2026-02-13   Bolton   16560000.0
17 2026-02-12   Bolton   12000000.0
18 2026-02-08   Bolton   13680000.0
19 2026-02-05   Bolton   12240000.0


In [8]:
conn.execute("""
    SELECT *
    FROM read_parquet('s3://year-in-data/tables/gymgroup_visits.parquet')
    ORDER BY date DESC
    LIMIT 20
""").df()

IOException: IO Error: Could not resolve hostname error for HTTP HEAD to 'https://tables.http://localhost%3A9000/gymgroup_visits.parquet'

LINE 3:     FROM read_parquet('s3://tables/gymgroup_visits.parquet')
                 ^

In [9]:
conn.execute("""
    SELECT *
    FROM read_parquet('s3://year-in-data/silver/gymgroup/visits.parquet')
    ORDER BY date DESC
    LIMIT 20
""").df()

,date,category,duration_ms
0,2026-04-01,Bolton,3480000.0
1,2026-03-27,Bolton,5220000.0
2,2026-03-25,Bolton,5580000.0
3,2026-03-23,Bolton,3300000.0
4,2026-03-19,Bolton,4140000.0
5,2026-03-17,Bolton,4260000.0
6,2026-03-08,Bolton,3960000.0
7,2026-03-06,Bolton,4740000.0
8,2026-03-05,Bolton,5160000.0
9,2026-02-27,Bolton,3180000.0
